In [ ]:
import os
import sys
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta, timezone

from Legacy.Binance_new import Binance # Get_Price_Data_By_Time

# import importlib
# import RL_Trade_POC_Util

# # Reload the module
# importlib.reload(RL_Trade_POC_Util)

from RL_Trade_POC_Util import normalize, get_level, get_change, get_volatility, plot_array, plot_array1, plot_arrays

In [ ]:
bn = Binance.instantiate(connect=False)

In [ ]:
# Get_Price_Data_By_Time function seamlessly collects/creates price candles from Binance.
datatype = "klines" # "aggTrades" #, "klines"
symbol = "BTCUSDT"      # Any pair listed on Binance.

# dates that worked
# start = datetime(2018, 1, 1)    # Any datetime. Note Binance started operation in July 2017.
# end = datetime(2026, 3, 26) # + timedelta(minutes=100)     # Any datetime

start = datetime(2026, 1, 1)    # Any datetime. Note Binance started operation in July 2017.
end = datetime(2026, 2, 26) # + timedelta(minutes=100)     # Any datetime

interval = '1m'    # in '1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '8h', '12h', '1d'.  (excludes: '3d', '1w')
print(f"interval: {interval}")
#------------------------------------------------------------------------------------------------------------------
candles, nFills, successful, _, reason = bn.Get_Price_Data_By_Time(datatype, symbol, interval, start, end)
print(reason)
if successful:          # price is a numpy array.
    print("\n Successful !!!!!!!!!! (nCandles, nAttributes) = {}, nCreated = {}".format(candles.shape, nFills))

In [ ]:
n = 500

x = candles[:n, 3]      # close
v = candles[:n, 4]      # volume
x_normal = normalize(x, axis=0) * 2
v_normal = normalize(v, axis=0) * 2
back = 30           # we look back as far as 'back'

In [ ]:

levels = []
for anchor in range(x.shape[0]):
    level = get_level(x, anchor, back, axis=0)
    levels.append(level)
levels = np.array(levels)
assert levels.shape == x.shape

plot_arrays([x_normal, levels], labels=['price_normalized', 'levels'], title="price levels", xlabel="time", ylabel="price", figsize=(18, 2), flip=False, save_path=None)

In [ ]:
price_changes = []
for anchor in range(x.shape[0]):
    pc = get_change(x, anchor, back, axis=0)
    price_changes.append(pc)
price_changes = np.array(price_changes)
assert price_changes.shape == x.shape

plot_arrays([x_normal, price_changes], labels=['price_normalized', 'price_changes'], title="price_changes", xlabel="time", ylabel="price", figsize=(18, 2), flip=False, save_path=None)


In [ ]:
levels = []
for anchor in range(x.shape[0]):
    level = get_level(v, anchor, back, axis=0)
    levels.append(level)
levels = np.array(levels)
assert levels.shape == x.shape

plot_arrays([v_normal, levels], labels=['volume_normalized', 'levels'], title="volume levels", xlabel="time", ylabel="price", figsize=(18, 2), flip=False, save_path=None)

In [ ]:
volume_changes = []
for anchor in range(v.shape[0]):
    vc = get_change(v, anchor, back, axis=0)
    volume_changes.append(vc)
volume_changes = np.array(volume_changes)
assert volume_changes.shape == v.shape

plot_arrays([v_normal, volume_changes], labels=['price_normalized', 'volume_changes'], title="volume_changes", xlabel="time", ylabel="price", figsize=(18, 2), flip=False, save_path=None)


In [ ]:
volatilities = []
for anchor in range(x.shape[0]):
    volatility = get_volatility(x, anchor, back, axis=0, span=back)
    volatilities.append(volatility)
volatilities = np.array(volatilities)
assert volatilities.shape == x.shape

plot_arrays([x_normal, volatilities], labels=['price_normalized', 'volatilities'], title="price volatilities", xlabel="time", ylabel="price", figsize=(18, 2), flip=False, save_path=None)


In [ ]:
from enum import Enum
import torch
import torch.nn as nn
import torch.nn.functional as F

class Action(Enum):
    Buy = -1
    Hold = 0
    Sell = 1
    
max_buys = 10
max_price_history = 60
state_features = max_price_history + 5 + max_buys
print(f"state_features {state_features}")
    
def get_reward(action, price_curr, buy_costs):
    would_sell_revenue = len(buy_costs) * price_curr
    total_cost = sum(buy_costs)
    would_sell_profit = would_sell_revenue - total_cost
    # print(f"would_sell_profit {would_sell_profit}")
    sell_reward = -1 if len(buy_costs) == 0 else (-1 if would_sell_profit <= 0 else 1)      # -1 if would_sell_profit <= 0 ---- think of fees
    dict = { Action.Buy: 0, Action.Hold: 0, Action.Sell: sell_reward}
    reward = dict[action]
    return reward

def get_state_tensor(prices, volumes, buy_costs):
    state = prices[- max_price_history :].tolist()     #----------------------- Consider introducing compressed history. No need to flip.
    assert len(state) == max_price_history
    state.append(get_level(prices, len(prices)-1, back, axis=0))
    state.append(get_change(prices, len(prices)-1, back, axis=0))
    state.append(get_level(volumes, len(prices)-1, back, axis=0))
    state.append(get_change(volumes, len(prices)-1, back, axis=0))
    state.append(get_volatility(prices, len(prices)-1, back, span=back))
    assert len(buy_costs) <= max_buys
    state = state + buy_costs + [0 for _ in range(max_buys - len(buy_costs))]
    state = torch.tensor(state)
    assert state.shape[-1] == state_features
    state = state.unsqueeze(0)  # (batch, state_features)
    return state

def act(action, prices, volumes, buy_costs):
    price_curr = prices[-1]
    reward = get_reward(action, price_curr, buy_costs)
    state = get_state_tensor(prices, volumes, buy_costs)
    done = action == Action.Sell

    cash_in, cash_out = 0, 0
    if action == Action.Buy:
        if len(buy_costs) < max_buys:
            cash_out += price_curr
            buy_costs.append(cash_out)
    elif action == Action.Sell:
        cash_in += len(buy_costs) * price_curr
        buy_costs = []
    
    assert cash_in <= 0 or cash_out <= 0
    net_cash_in = cash_in - cash_out
    
    state = get_state_tensor(prices, volumes, buy_costs)
    
    return state, reward, done, buy_costs, net_cash_in

In [ ]:
import random

buy_costs = []
actions = [Action.Buy] * 3 + [Action.Hold] * 10 + [Action.Sell] * 1

reward_history = []
inventory_value_history = []
cash_balance_history, cash_balance = [], 0

safe_init_anchor = max(3 * back - 2, max_price_history - 1)
max_history_required = safe_init_anchor

for anchor in range(len(x)):  
    if anchor < safe_init_anchor: continue
    
    action = random.choice(actions)
    prices = x [anchor - max_history_required + 1 : anchor + 1]
    volumes = v [anchor - max_history_required + 1 : anchor + 1]
    
    price_curr = prices[-1]
    state, reward, done, buy_costs, net_cash_in = act(action, prices, volumes, buy_costs)
    reward_history.append(reward)
    inventory_value_history.append(len(buy_costs)*price_curr)
    cash_balance += net_cash_in
    cash_balance_history.append(cash_balance)
    
    # print(action, f"reward/cash_in/balance {reward}/{net_cash_in}/{cash_balance} {'positive' if cash_balance > 0 else 'nagative'}")

cash_balance_history = np.array(cash_balance_history)
cash_balance_history = cash_balance_history / cash_balance_history.std() * 7

inventory_value_history = np.array(inventory_value_history)
inventory_value_history = inventory_value_history/inventory_value_history.std() * 7

total_balance_history = (cash_balance_history + inventory_value_history) * 300
    
arrays = [x_normal[safe_init_anchor : ] * 10, np.array(reward_history) * 5, inventory_value_history, cash_balance_history, total_balance_history]
labels = ['prices', 'reward', 'inventory_value', 'cash_balance', 'total_balance']
length = len(x_normal[safe_init_anchor : ])
for arr in arrays: assert len(arr) == length
plot_arrays(arrays, labels, title="action history - all scaled", xlabel="", ylabel="", figsize=(18, 4), flip=False, lw=0.5)

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
        
    def forward(self, x, ss=None):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return x
    
class ActionValue(nn.Module):
    def __init__(self, state_features, action_features, hidden_features):
        super().__init__()
        self.ffn = PositionwiseFeedForward(state_features + action_features, hidden_features, out_features=1, act_layer=nn.GELU)
        
    def forward(self, state, action):
        input = torch.concat([state, action], dim=-1)
        ffn_out = self.ffn(input)      # scaled for faster training. expected value in {-1, +1}
        return ffn_out

Q = ActionValue(20, 1, 50)
state = torch.tensor([1.2] * 20).unsqueeze(0)
action = torch.tensor([1]).unsqueeze(0)
with torch.set_grad_enabled(False):
    q = Q(state, action).squeeze(0)
print(q)

In [ ]:
import random
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move your Q-network to GPU
Q = ActionValue(state_features, action_features=1, hidden_features=state_features * 4).to(device)

action_embeddings = [action.value for action in Action]
actions_list = list(Action)

def get_Q_values(Q_model, state): # state: (batch, state_features)
    q_values = []
    
    with torch.no_grad():  # Fixed: torch.set_grad_enabled(False) is deprecated, use torch.no_grad()
        for action_emb in action_embeddings:
            action_tensor = torch.tensor([action_emb]).unsqueeze(0).repeat(state.shape[0], 1).to(device)
            q_values.append(Q_model(state, action_tensor))
    q_values = torch.concat(q_values, dim=-1)
    assert q_values.shape == (state.shape[0], len(actions_list))
    return q_values # (batch_size, len(action_list))

def get_best_value(Q_model, state):
    q_values_tensor = get_Q_values(Q_model, state)  # (batch_size, len(action_list))
    best_value = q_values_tensor.max(dim=-1).values
    return best_value   # (batch_size,)

def get_best_action(Q_model, state):
    q_values_tensor = get_Q_values(Q_model, state)  # (batch_size, len(action_list))
    argmax = torch.argmax(q_values_tensor, dim=-1)   # (batch_size,)
    actions = [actions_list[argmax[idx].item()] for idx in range(argmax.shape[0])]
    return actions

def train(nEpochs, epsilon, gamma, batch_size):
    
    trainable_params = [p for p in Q.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-5)
    
    memory, max_memory = [], 100
    
    close_idx, volume_idx = 3, 4
    env_len = len(candles[:, close_idx])
    
    for epoch in range(nEpochs):
        epoch_epsilon = max(epsilon * (1 - epoch / nEpochs), 0.05)
        
        epoch_span = 300
        anchor_curr = random.randint(safe_init_anchor, env_len - epoch_span)

        buy_costs = []
        prices = candles[: anchor_curr + 1, close_idx]
        volumes = candles[: anchor_curr + 1, volume_idx]
        state = get_state_tensor(prices, volumes, buy_costs).to(device)  # Move state to GPU
        
        done = False; nActions = 0
        while not done:
            
            #------------- decide action to take ---------------
            regular_action = get_best_action(Q, state)[0]
            r = random.randint(1, 100)
            random_action = Action.Buy if r <= 30 else Action.Hold if r <= 90 else Action.Sell
            action = random_action if random.randint(1, 100) < epoch_epsilon * 100 else regular_action
            action_type = action
 
            #-------------- act the action on the next env ----------           
            anchor_curr += 1
            prices = candles[: anchor_curr + 1, close_idx]
            volumes = candles[: anchor_curr + 1, volume_idx]
            state_new, reward, done, buy_costs, _ = act(action, prices, volumes, buy_costs)
            
            # Move state_new to GPU and convert action to tensor
            state_new = state_new.to(device)
            action_tensor = torch.tensor([action.value]).unsqueeze(0).to(device)
            
            memory.append((state, action_tensor, reward, state_new, done))
            memory = memory[-max_memory:]
            state = state_new
            nActions += 1
            
            #-------------- sample mini-batch from memory ------------
            steps_sparse = random.sample(memory, min(batch_size, len(memory)))            
            steps_sparse = [(state, action, reward + (gamma * get_best_value(Q, state_new)[0].item() if not done else 0)) 
                           for state, action, reward, state_new, done in memory]
            
            with torch.set_grad_enabled(True):
                optimizer.zero_grad()  # Important: Clear gradients before backward
                for (state, action, action_value_true) in steps_sparse:
                    action_value_true = torch.tensor([action_value_true]).to(device)
                    action_value_pred = Q(state, action).squeeze(0)
                    loss = torch.sqrt(torch.square(action_value_pred - action_value_true).mean())
                    loss.backward()
                optimizer.step()
                
            print(f"epoch {epoch}, nActions {nActions}", f"regular/random/action {regular_action}/{random_action}/{action_type}", f"loss {loss.item()}")
            
        print("-" * 80, f"epoch {epoch}, nActions {nActions}")

In [ ]:
train(nEpochs=30000, epsilon=0.5, gamma=1, batch_size=10)
print(f"Done")